# Experimentos de Pricing: Logistic Regression Pointwise e Agregada

Notebook para testar modelos simples de **Regressão Logística** no problema de simulação → proposta.

Estrutura:

1. Começa com uma base **agregada**: 1 linha por jornada/cliente.
2. Depois avança para base **pointwise completa**: 1 linha por simulação.
3. Cria somente features de **lag 1 a 4** para:
   - `vl_financiado`
   - `vl_entrada`
   - `pct_entrada`
   - `tp_gh`
4. Usa também as categóricas/base:
   - `fl_zero`
   - `numero_ano_mod_veic`
5. Compara regressão logística com:
   - baseline OneHot + StandardScaler
   - WOE simples
   - OptBinning + WOE
   - RandomTreesEmbedding / TreeEncoder
   - TreeBinning + WOE
   - Splines
   - Polinômio grau 2
6. Compara pesos:
   - sem peso
   - `class_weight='balanced'`
   - `peso_final = peso_classe * peso_jornada`
7. Avalia treino, teste e out-of-time com ROC-AUC, Gini, PR-AUC, KS e Brier, além de calibração e coeficientes.

> Ajuste a seção **Configuração** antes de executar.


## 0. Instalação opcional

In [ ]:
# %pip install optbinning scikit-learn pandas numpy matplotlib scipy
# dbutils.library.restartPython()


## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures, SplineTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomTreesEmbedding
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.utils.class_weight import compute_sample_weight

RANDOM_STATE = 42
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## 2. Configuração

Ajuste os nomes de colunas para sua base real. O notebook assume que existe uma coluna de target pointwise, isto é, `1` se **aquela simulação** virou proposta.

In [ ]:
DATA_PATH = '/mnt/data/sua_base.parquet'
FILE_FORMAT = 'parquet'  # 'parquet', 'csv' ou 'delta_spark'

ID_CLIENTE_COL = 'id_cliente'
ID_JORNADA_COL = 'id_jornada'
ID_SIMULACAO_COL = 'id_simulacao'
DT_SIMULACAO_COL = 'dt_simulacao'
TARGET_COL = 'target_proposta'

LAG_FEATURES = ['vl_financiado', 'vl_entrada', 'pct_entrada', 'tp_gh']
BASE_FEATURES = ['vl_financiado', 'vl_entrada', 'pct_entrada', 'tp_gh', 'fl_zero', 'numero_ano_mod_veic']
CATEGORICAL_FEATURES = ['tp_gh', 'fl_zero', 'numero_ano_mod_veic']
NUMERIC_FEATURES = ['vl_financiado', 'vl_entrada', 'pct_entrada']
MAX_LAG = 4

# Split temporal automático por quantis da data de início da jornada
USE_AUTO_TEMPORAL_SPLIT = True
TRAIN_Q = 0.70
TEST_Q = 0.85

# Split temporal manual caso USE_AUTO_TEMPORAL_SPLIT=False
TEST_START_DATE = '2025-01-01'
OOT_START_DATE = '2025-04-01'


## 3. Carregamento dos dados

In [ ]:
def load_dataframe(path: str, file_format: str = 'parquet') -> pd.DataFrame:
    if file_format == 'parquet':
        return pd.read_parquet(path)
    if file_format == 'csv':
        return pd.read_csv(path)
    if file_format == 'delta_spark':
        sdf = spark.read.format('delta').load(path)  # noqa: F821
        return sdf.toPandas()
    raise ValueError(f'Formato não suportado: {file_format}')

# df_raw = load_dataframe(DATA_PATH, FILE_FORMAT)
# display(df_raw.head())


## 4. Validação de colunas

In [ ]:
def check_required_columns(df: pd.DataFrame):
    required = [ID_CLIENTE_COL, ID_JORNADA_COL, ID_SIMULACAO_COL, DT_SIMULACAO_COL, TARGET_COL] + BASE_FEATURES
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Colunas ausentes na base: {missing}')
    print('Todas as colunas obrigatórias foram encontradas.')

# check_required_columns(df_raw)


## 5. Feature engineering: lags e categóricas simples

Apenas lags das variáveis solicitadas são criados. As features acumuladas/tendências ficam fora por enquanto, para manter o experimento simples.

In [ ]:
def prepare_base_types(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[DT_SIMULACAO_COL] = pd.to_datetime(df[DT_SIMULACAO_COL])
    df[TARGET_COL] = df[TARGET_COL].astype(int)
    for col in CATEGORICAL_FEATURES:
        if col in df.columns:
            df[col] = df[col].astype('string').fillna('__MISSING__')
    for col in NUMERIC_FEATURES:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def cut_after_first_proposal(df: pd.DataFrame) -> pd.DataFrame:
    # Para um modelo de primeira proposta, remove simulações posteriores à primeira proposta.
    df = df.sort_values([ID_JORNADA_COL, DT_SIMULACAO_COL, ID_SIMULACAO_COL]).copy()
    df['ordem_simulacao'] = df.groupby(ID_JORNADA_COL).cumcount() + 1
    first_event_order = (
        df.loc[df[TARGET_COL] == 1, [ID_JORNADA_COL, 'ordem_simulacao']]
        .groupby(ID_JORNADA_COL)['ordem_simulacao']
        .min()
        .rename('ordem_primeira_proposta')
    )
    df = df.merge(first_event_order, on=ID_JORNADA_COL, how='left')
    mask = df['ordem_primeira_proposta'].isna() | (df['ordem_simulacao'] <= df['ordem_primeira_proposta'])
    return df.loc[mask].copy()


def add_lag_features(df: pd.DataFrame, max_lag: int = 4) -> pd.DataFrame:
    df = df.sort_values([ID_JORNADA_COL, DT_SIMULACAO_COL, ID_SIMULACAO_COL]).copy()
    df['ordem_simulacao'] = df.groupby(ID_JORNADA_COL).cumcount() + 1
    for col in LAG_FEATURES:
        for lag in range(1, max_lag + 1):
            df[f'{col}_lag_{lag}'] = df.groupby(ID_JORNADA_COL)[col].shift(lag)
    return df


def add_simple_categorical_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if {'fl_zero', 'numero_ano_mod_veic'}.issubset(df.columns):
        df['fl_zero_x_ano_mod'] = df['fl_zero'].astype('string').fillna('__MISSING__') + '__' + df['numero_ano_mod_veic'].astype('string').fillna('__MISSING__')
    if {'tp_gh', 'fl_zero'}.issubset(df.columns):
        df['tp_gh_x_fl_zero'] = df['tp_gh'].astype('string').fillna('__MISSING__') + '__' + df['fl_zero'].astype('string').fillna('__MISSING__')
    return df


def add_journey_weights_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    qtd = df.groupby(ID_JORNADA_COL).size().rename('qtd_simulacoes_jornada')
    df = df.merge(qtd, on=ID_JORNADA_COL, how='left')
    df['peso_jornada'] = 1.0 / df['qtd_simulacoes_jornada'].clip(lower=1)
    return df


## 6. Construção das bases: pointwise e agregada

In [ ]:
def build_pointwise_dataset(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = prepare_base_types(df_raw)
    df = cut_after_first_proposal(df)
    df = add_lag_features(df, max_lag=MAX_LAG)
    df = add_simple_categorical_features(df)
    df = add_journey_weights_columns(df)
    return df


def build_aggregated_dataset(df_pointwise: pd.DataFrame) -> pd.DataFrame:
    # Agregado: 1 linha por jornada, usando a última simulação observada até a primeira proposta ou fim da jornada.
    df = df_pointwise.sort_values([ID_JORNADA_COL, DT_SIMULACAO_COL, ID_SIMULACAO_COL]).copy()
    target_jornada = df.groupby(ID_JORNADA_COL)[TARGET_COL].max().rename('target_jornada')
    last_rows = df.groupby(ID_JORNADA_COL).tail(1).copy()
    last_rows = last_rows.drop(columns=[TARGET_COL], errors='ignore')
    last_rows = last_rows.merge(target_jornada, on=ID_JORNADA_COL, how='left')
    last_rows[TARGET_COL] = last_rows['target_jornada'].astype(int)
    last_rows = last_rows.drop(columns=['target_jornada'])
    last_rows['peso_jornada'] = 1.0
    return last_rows

# df_pointwise = build_pointwise_dataset(df_raw)
# df_agg = build_aggregated_dataset(df_pointwise)
# display(df_pointwise.head())
# display(df_agg.head())


## 7. Seleção de features

In [ ]:
def get_feature_lists(df: pd.DataFrame):
    lag_cols = [f'{col}_lag_{lag}' for col in LAG_FEATURES for lag in range(1, MAX_LAG + 1)]
    simple_cat_cols = [c for c in ['fl_zero_x_ano_mod', 'tp_gh_x_fl_zero'] if c in df.columns]
    categorical_cols, numeric_cols = [], []
    for c in BASE_FEATURES + lag_cols + simple_cat_cols:
        if c not in df.columns:
            continue
        if c in CATEGORICAL_FEATURES or c.startswith('tp_gh_lag') or c in simple_cat_cols:
            categorical_cols.append(c)
        else:
            numeric_cols.append(c)
    if 'ordem_simulacao' in df.columns:
        numeric_cols.append('ordem_simulacao')
    categorical_cols = list(dict.fromkeys(categorical_cols))
    numeric_cols = list(dict.fromkeys(numeric_cols))
    return numeric_cols, categorical_cols, numeric_cols + categorical_cols

# numeric_cols_agg, categorical_cols_agg, features_agg = get_feature_lists(df_agg)
# numeric_cols_pw, categorical_cols_pw, features_pw = get_feature_lists(df_pointwise)


## 8. Split treino, teste e OOT

O split é por data de início da jornada, evitando colocar simulações da mesma jornada em splits diferentes.

In [ ]:
def assign_temporal_split(df: pd.DataFrame, auto: bool = True) -> pd.DataFrame:
    df = df.copy()
    start_dates = df.groupby(ID_JORNADA_COL)[DT_SIMULACAO_COL].min().rename('dt_inicio_jornada')
    df = df.merge(start_dates, on=ID_JORNADA_COL, how='left')
    if auto:
        q_train = df['dt_inicio_jornada'].quantile(TRAIN_Q)
        q_test = df['dt_inicio_jornada'].quantile(TEST_Q)
        print(f'Corte treino/teste automático: {q_train}')
        print(f'Corte teste/OOT automático: {q_test}')
        df['split'] = np.where(df['dt_inicio_jornada'] <= q_train, 'train', np.where(df['dt_inicio_jornada'] <= q_test, 'test', 'oot'))
    else:
        test_start = pd.to_datetime(TEST_START_DATE)
        oot_start = pd.to_datetime(OOT_START_DATE)
        df['split'] = np.where(df['dt_inicio_jornada'] < test_start, 'train', np.where(df['dt_inicio_jornada'] < oot_start, 'test', 'oot'))
    return df

# df_agg_split = assign_temporal_split(df_agg, auto=USE_AUTO_TEMPORAL_SPLIT)
# df_pw_split = assign_temporal_split(df_pointwise, auto=USE_AUTO_TEMPORAL_SPLIT)
# display(df_pw_split.groupby('split')[TARGET_COL].agg(['count', 'mean']))


## 9. Métricas

In [ ]:
def ks_score(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    mask = ~pd.isna(y_score)
    y_true, y_score = y_true[mask], y_score[mask]
    if len(np.unique(y_true)) < 2:
        return np.nan
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return np.max(tpr - fpr)


def classification_metrics(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return {'roc_auc': np.nan, 'gini': np.nan, 'pr_auc': np.nan, 'ks': np.nan, 'brier': np.nan}
    roc = roc_auc_score(y_true, y_score)
    return {
        'roc_auc': roc,
        'gini': 2 * roc - 1,
        'pr_auc': average_precision_score(y_true, y_score),
        'ks': ks_score(y_true, y_score),
        'brier': brier_score_loss(y_true, y_score),
    }


def evaluate_model_by_split(model, df, features, model_name, dataset_name):
    rows = []
    keep_cols = [ID_JORNADA_COL, TARGET_COL, 'split']
    if ID_SIMULACAO_COL in df.columns:
        keep_cols.append(ID_SIMULACAO_COL)
    if 'ordem_simulacao' in df.columns:
        keep_cols.append('ordem_simulacao')
    pred_df = df[keep_cols].copy()
    pred_df['model_name'] = model_name
    pred_df['dataset_name'] = dataset_name
    pred_df['score'] = np.nan
    for split_name in ['train', 'test', 'oot']:
        mask = df['split'] == split_name
        if mask.sum() == 0:
            continue
        X = df.loc[mask, features]
        y = df.loc[mask, TARGET_COL].astype(int)
        score = model.predict_proba(X)[:, 1]
        pred_df.loc[mask, 'score'] = score
        rows.append({'dataset_name': dataset_name, 'model_name': model_name, 'split': split_name, 'n': len(y), 'event_rate': y.mean(), **classification_metrics(y, score)})
    return pd.DataFrame(rows), pred_df


## 10. Transformers: WOE, TreeBinning + WOE e OptBinning

In [ ]:
class SimpleWOETransformer(BaseEstimator, TransformerMixin):
    def __init__(self, numeric_cols=None, categorical_cols=None, n_bins=10, smoothing=0.5):
        self.numeric_cols = numeric_cols or []
        self.categorical_cols = categorical_cols or []
        self.n_bins = n_bins
        self.smoothing = smoothing

    def _calc_woe(self, event, nonevent, total_event, total_nonevent):
        event_rate = (event + self.smoothing) / (total_event + 2 * self.smoothing)
        nonevent_rate = (nonevent + self.smoothing) / (total_nonevent + 2 * self.smoothing)
        return np.log(event_rate / nonevent_rate)

    def _fit_map(self, s, y, total_event, total_nonevent):
        tmp = pd.DataFrame({'bin': s.astype(str), 'y': y})
        stats = tmp.groupby('bin')['y'].agg(['sum', 'count'])
        stats['nonevent'] = stats['count'] - stats['sum']
        stats['woe'] = [self._calc_woe(ev, nev, total_event, total_nonevent) for ev, nev in zip(stats['sum'], stats['nonevent'])]
        return stats['woe'].to_dict()

    def fit(self, X, y):
        X = pd.DataFrame(X).copy().reset_index(drop=True)
        y = pd.Series(y).astype(int).reset_index(drop=True)
        self.maps_, self.bin_edges_ = {}, {}
        total_event, total_nonevent = y.sum(), len(y) - y.sum()
        self.global_woe_ = self._calc_woe(total_event, total_nonevent, total_event, total_nonevent)
        for col in self.numeric_cols:
            s = pd.to_numeric(X[col], errors='coerce')
            try:
                _, bins = pd.qcut(s, q=self.n_bins, retbins=True, duplicates='drop')
                bins = np.unique(bins)
            except Exception:
                bins = np.array([-np.inf, np.inf])
            if len(bins) < 2 or pd.isna(bins).any():
                bins = np.array([-np.inf, np.inf])
            else:
                bins[0], bins[-1] = -np.inf, np.inf
            self.bin_edges_[col] = bins
            binned = pd.cut(s, bins=bins, include_lowest=True).astype(str).fillna('__MISSING__')
            self.maps_[col] = self._fit_map(binned, y, total_event, total_nonevent)
        for col in self.categorical_cols:
            s = X[col].astype('string').fillna('__MISSING__')
            self.maps_[col] = self._fit_map(s, y, total_event, total_nonevent)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = pd.DataFrame(index=X.index)
        for col in self.numeric_cols:
            s = pd.to_numeric(X[col], errors='coerce')
            binned = pd.cut(s, bins=self.bin_edges_[col], include_lowest=True).astype(str).fillna('__MISSING__')
            out[f'{col}__woe'] = binned.map(self.maps_[col]).fillna(self.global_woe_).astype(float)
        for col in self.categorical_cols:
            s = X[col].astype('string').fillna('__MISSING__')
            out[f'{col}__woe'] = s.map(self.maps_[col]).fillna(self.global_woe_).astype(float)
        return out

    def get_feature_names_out(self, input_features=None):
        return np.array([f'{c}__woe' for c in self.numeric_cols + self.categorical_cols])


class TreeBinningWOETransformer(SimpleWOETransformer):
    def __init__(self, numeric_cols=None, categorical_cols=None, max_leaf_nodes=6, min_samples_leaf=0.03, smoothing=0.5):
        super().__init__(numeric_cols=numeric_cols, categorical_cols=categorical_cols, smoothing=smoothing)
        self.max_leaf_nodes = max_leaf_nodes
        self.min_samples_leaf = min_samples_leaf

    def fit(self, X, y):
        X = pd.DataFrame(X).copy().reset_index(drop=True)
        y = pd.Series(y).astype(int).reset_index(drop=True)
        self.trees_, self.maps_ = {}, {}
        total_event, total_nonevent = y.sum(), len(y) - y.sum()
        self.global_woe_ = self._calc_woe(total_event, total_nonevent, total_event, total_nonevent)
        for col in self.numeric_cols:
            s = pd.to_numeric(X[col], errors='coerce')
            med = s.median() if not pd.isna(s.median()) else 0
            arr = s.fillna(med).to_numpy().reshape(-1, 1)
            tree = DecisionTreeClassifier(max_leaf_nodes=self.max_leaf_nodes, min_samples_leaf=self.min_samples_leaf, random_state=RANDOM_STATE)
            tree.fit(arr, y)
            leaf = pd.Series(tree.apply(arr).astype(str))
            self.trees_[col] = tree
            self.maps_[col] = self._fit_map(leaf, y, total_event, total_nonevent)
        for col in self.categorical_cols:
            s = X[col].astype('string').fillna('__MISSING__')
            self.maps_[col] = self._fit_map(s, y, total_event, total_nonevent)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = pd.DataFrame(index=X.index)
        for col in self.numeric_cols:
            s = pd.to_numeric(X[col], errors='coerce')
            med = s.median() if not pd.isna(s.median()) else 0
            leaf = pd.Series(self.trees_[col].apply(s.fillna(med).to_numpy().reshape(-1, 1)).astype(str), index=X.index)
            out[f'{col}__tree_woe'] = leaf.map(self.maps_[col]).fillna(self.global_woe_).astype(float)
        for col in self.categorical_cols:
            s = X[col].astype('string').fillna('__MISSING__')
            out[f'{col}__woe'] = s.map(self.maps_[col]).fillna(self.global_woe_).astype(float)
        return out

    def get_feature_names_out(self, input_features=None):
        return np.array([f'{c}__tree_woe' for c in self.numeric_cols] + [f'{c}__woe' for c in self.categorical_cols])


class OptBinningWOETransformer(BaseEstimator, TransformerMixin):
    def __init__(self, numeric_cols=None, categorical_cols=None, max_n_bins=6):
        self.numeric_cols = numeric_cols or []
        self.categorical_cols = categorical_cols or []
        self.max_n_bins = max_n_bins

    def fit(self, X, y):
        try:
            from optbinning import OptimalBinning
        except Exception as e:
            raise ImportError('Instale optbinning para usar este transformer: %pip install optbinning') from e
        X = pd.DataFrame(X).copy().reset_index(drop=True)
        y = pd.Series(y).astype(int).reset_index(drop=True)
        self.binners_, self.feature_names_ = {}, []
        for col in self.numeric_cols:
            optb = OptimalBinning(name=col, dtype='numerical', max_n_bins=self.max_n_bins, solver='cp')
            optb.fit(pd.to_numeric(X[col], errors='coerce').values, y.values)
            self.binners_[col] = optb
            self.feature_names_.append(f'{col}__optb_woe')
        for col in self.categorical_cols:
            optb = OptimalBinning(name=col, dtype='categorical', max_n_bins=self.max_n_bins, solver='cp')
            optb.fit(X[col].astype(str).fillna('__MISSING__').values, y.values)
            self.binners_[col] = optb
            self.feature_names_.append(f'{col}__optb_woe')
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = pd.DataFrame(index=X.index)
        for col in self.numeric_cols:
            out[f'{col}__optb_woe'] = self.binners_[col].transform(pd.to_numeric(X[col], errors='coerce').values, metric='woe')
        for col in self.categorical_cols:
            out[f'{col}__optb_woe'] = self.binners_[col].transform(X[col].astype(str).fillna('__MISSING__').values, metric='woe')
        return out.astype(float)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_)


## 11. Pipelines de modelagem

In [ ]:
def make_logistic(class_weight=None):
    return LogisticRegression(max_iter=2000, solver='lbfgs', class_weight=class_weight, random_state=RANDOM_STATE)


def make_baseline_pipeline(numeric_cols, categorical_cols, class_weight=None):
    pre = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')), ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01))]), categorical_cols),
    ])
    return Pipeline([('preprocess', pre), ('model', make_logistic(class_weight))])


def make_woe_pipeline(numeric_cols, categorical_cols, class_weight=None):
    return Pipeline([('woe', SimpleWOETransformer(numeric_cols, categorical_cols, n_bins=10)), ('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', make_logistic(class_weight))])


def make_optbinning_pipeline(numeric_cols, categorical_cols, class_weight=None):
    return Pipeline([('optbinning', OptBinningWOETransformer(numeric_cols, categorical_cols, max_n_bins=6)), ('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', make_logistic(class_weight))])


def make_treeencoder_pipeline(numeric_cols, categorical_cols, class_weight=None):
    pre = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')), ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01))]), categorical_cols),
    ])
    return Pipeline([
        ('base_preprocess', pre),
        ('tree_encoder', RandomTreesEmbedding(n_estimators=80, max_depth=3, min_samples_leaf=50, random_state=RANDOM_STATE, sparse_output=True)),
        ('model', make_logistic(class_weight)),
    ])


def make_treebinning_woe_pipeline(numeric_cols, categorical_cols, class_weight=None):
    return Pipeline([('treebin_woe', TreeBinningWOETransformer(numeric_cols, categorical_cols, max_leaf_nodes=6)), ('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', make_logistic(class_weight))])


def make_spline_pipeline(numeric_cols, categorical_cols, class_weight=None):
    pre = ColumnTransformer([
        ('num_spline', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('spline', SplineTransformer(n_knots=5, degree=3, include_bias=False))]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')), ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01))]), categorical_cols),
    ])
    return Pipeline([('preprocess', pre), ('model', make_logistic(class_weight))])


def make_poly2_pipeline(numeric_cols, categorical_cols, class_weight=None):
    pre = ColumnTransformer([
        ('num_poly', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('poly', PolynomialFeatures(degree=2, include_bias=False))]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')), ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01))]), categorical_cols),
    ])
    return Pipeline([('preprocess', pre), ('model', make_logistic(class_weight))])


def get_model_factories():
    return {
        'logreg_baseline': make_baseline_pipeline,
        'logreg_woe': make_woe_pipeline,
        'logreg_optbinning': make_optbinning_pipeline,
        'logreg_treeencoder': make_treeencoder_pipeline,
        'logreg_treebinning_woe': make_treebinning_woe_pipeline,
        'logreg_splines': make_spline_pipeline,
        'logreg_poly2': make_poly2_pipeline,
    }


## 12. Pesos: classe, jornada e peso final

In [ ]:
def add_class_and_final_weights(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    train_mask = df['split'] == 'train'
    y_train = df.loc[train_mask, TARGET_COL].astype(int)
    class_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)
    class_weight_map = pd.DataFrame({'y': y_train.values, 'w': class_weights_train}).groupby('y')['w'].mean().to_dict()
    df['peso_classe'] = df[TARGET_COL].map(class_weight_map).fillna(1.0).astype(float)
    df['peso_final'] = df['peso_classe'] * df['peso_jornada'].fillna(1.0)
    return df

# df_agg_split = add_class_and_final_weights(df_agg_split)
# df_pw_split = add_class_and_final_weights(df_pw_split)


## 13. Loop de treinamento e avaliação

In [ ]:
def fit_one_model(factory, df, features, numeric_cols, categorical_cols, weight_strategy):
    train = df[df['split'] == 'train'].copy()
    X_train = train[features]
    y_train = train[TARGET_COL].astype(int)
    if weight_strategy == 'none':
        model, sample_weight = factory(numeric_cols, categorical_cols, class_weight=None), None
    elif weight_strategy == 'class_weight_balanced':
        model, sample_weight = factory(numeric_cols, categorical_cols, class_weight='balanced'), None
    elif weight_strategy == 'peso_classe_x_peso_jornada':
        model, sample_weight = factory(numeric_cols, categorical_cols, class_weight=None), train['peso_final'].values
    else:
        raise ValueError(weight_strategy)
    if sample_weight is None:
        model.fit(X_train, y_train)
    else:
        model.fit(X_train, y_train, model__sample_weight=sample_weight)
    return model


def run_experiments(df, dataset_name):
    numeric_cols, categorical_cols, features = get_feature_lists(df)
    factories = get_model_factories()
    weight_strategies = ['none', 'class_weight_balanced', 'peso_classe_x_peso_jornada']
    all_metrics, all_predictions, fitted_models = [], [], {}
    for weight_strategy in weight_strategies:
        for model_label, factory in factories.items():
            full_name = f'{model_label}__{weight_strategy}'
            print(f'Treinando: {dataset_name} | {full_name}')
            try:
                model = fit_one_model(factory, df, features, numeric_cols, categorical_cols, weight_strategy)
                metrics_df, pred_df = evaluate_model_by_split(model, df, features, full_name, dataset_name)
                all_metrics.append(metrics_df)
                all_predictions.append(pred_df)
                fitted_models[(dataset_name, full_name)] = model
            except Exception as e:
                print(f'Falhou: {dataset_name} | {full_name} | erro: {e}')
    metrics = pd.concat(all_metrics, ignore_index=True) if all_metrics else pd.DataFrame()
    predictions = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
    return metrics, predictions, fitted_models

# metrics_agg, pred_agg, models_agg = run_experiments(df_agg_split, dataset_name='agregado')
# metrics_pw, pred_pw, models_pw = run_experiments(df_pw_split, dataset_name='pointwise')
# metrics_all = pd.concat([metrics_agg, metrics_pw], ignore_index=True)
# pred_all = pd.concat([pred_agg, pred_pw], ignore_index=True)
# display(metrics_all.sort_values(['dataset_name', 'split', 'roc_auc'], ascending=[True, True, False]))


## 14. Comparação de métricas por split

In [ ]:
def pivot_metric_table(metrics_all, metric='roc_auc'):
    return (
        metrics_all
        .pivot_table(index=['dataset_name', 'model_name'], columns='split', values=metric, aggfunc='mean')
        .reset_index()
        .sort_values(['dataset_name', 'oot'], ascending=[True, False])
    )

# for metric in ['roc_auc', 'gini', 'pr_auc', 'ks', 'brier']:
#     print(metric)
#     display(pivot_metric_table(metrics_all, metric))


## 15. Gráficos de comparação no OOT

In [ ]:
def plot_oot_metric(metrics_all, metric='roc_auc', top_n=20):
    dfp = metrics_all[metrics_all['split'] == 'oot'].copy().sort_values(metric, ascending=False).head(top_n)
    plt.figure(figsize=(12, 6))
    plt.barh(dfp['dataset_name'] + ' | ' + dfp['model_name'], dfp[metric])
    plt.gca().invert_yaxis()
    plt.title(f'Top {top_n} modelos por {metric} no OOT')
    plt.xlabel(metric)
    plt.tight_layout()
    plt.show()

# plot_oot_metric(metrics_all, 'roc_auc')
# plot_oot_metric(metrics_all, 'pr_auc')
# plot_oot_metric(metrics_all, 'ks')
# plot_oot_metric(metrics_all, 'brier')


## 16. Calibração

In [ ]:
def plot_calibration_curves(predictions, dataset_name, split='oot', top_models=None, n_bins=10):
    dfp = predictions[(predictions['dataset_name'] == dataset_name) & (predictions['split'] == split)].copy()
    if top_models is not None:
        dfp = dfp[dfp['model_name'].isin(top_models)]
    plt.figure(figsize=(8, 8))
    plt.plot([0, 1], [0, 1], linestyle='--', label='Perfeitamente calibrado')
    for model_name, g in dfp.groupby('model_name'):
        g = g.dropna(subset=['score'])
        if g[TARGET_COL].nunique() < 2 or len(g) < n_bins:
            continue
        frac_pos, mean_pred = calibration_curve(g[TARGET_COL].astype(int), g['score'], n_bins=n_bins, strategy='quantile')
        plt.plot(mean_pred, frac_pos, marker='o', label=model_name[:70])
    plt.title(f'Curvas de calibração | {dataset_name} | {split}')
    plt.xlabel('Probabilidade média predita')
    plt.ylabel('Frequência observada')
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

# top_agg = metrics_all[(metrics_all.dataset_name == 'agregado') & (metrics_all.split == 'oot')].sort_values('roc_auc', ascending=False).head(5)['model_name'].tolist()
# top_pw = metrics_all[(metrics_all.dataset_name == 'pointwise') & (metrics_all.split == 'oot')].sort_values('roc_auc', ascending=False).head(5)['model_name'].tolist()
# plot_calibration_curves(pred_all, 'agregado', 'oot', top_agg)
# plot_calibration_curves(pred_all, 'pointwise', 'oot', top_pw)


## 17. Análise por decil de score

In [ ]:
def decile_analysis(predictions, dataset_name, model_name, split='oot', n_bins=10):
    dfp = predictions[(predictions['dataset_name'] == dataset_name) & (predictions['model_name'] == model_name) & (predictions['split'] == split)].dropna(subset=['score']).copy()
    dfp['decil'] = pd.qcut(dfp['score'], q=n_bins, labels=False, duplicates='drop') + 1
    return (
        dfp.groupby('decil')
        .agg(n=(TARGET_COL, 'size'), event_rate=(TARGET_COL, 'mean'), score_min=('score', 'min'), score_mean=('score', 'mean'), score_max=('score', 'max'))
        .reset_index()
        .sort_values('decil', ascending=False)
    )

# best_row = metrics_all[metrics_all['split'] == 'oot'].sort_values('roc_auc', ascending=False).iloc[0]
# display(decile_analysis(pred_all, best_row.dataset_name, best_row.model_name, 'oot'))


## 18. Diagnóstico da hipótese de jornada no pointwise

Verifica se, nas jornadas convertidas, a simulação que virou proposta ficou com score maior que as anteriores.

In [ ]:
def journey_ranking_diagnostics(df_pointwise_split, predictions, model_name, split='oot'):
    pred = predictions[(predictions['dataset_name'] == 'pointwise') & (predictions['model_name'] == model_name) & (predictions['split'] == split)].copy()
    tmp = df_pointwise_split[df_pointwise_split['split'] == split][[ID_JORNADA_COL, ID_SIMULACAO_COL, 'ordem_simulacao', TARGET_COL]].copy().reset_index(drop=True)
    tmp['score'] = pred['score'].reset_index(drop=True)
    conv_journeys = tmp.groupby(ID_JORNADA_COL)[TARGET_COL].max()
    conv_journeys = conv_journeys[conv_journeys == 1].index
    tmp = tmp[tmp[ID_JORNADA_COL].isin(conv_journeys)].copy()
    rows = []
    for jid, g in tmp.groupby(ID_JORNADA_COL):
        g = g.sort_values('ordem_simulacao')
        event_rows = g[g[TARGET_COL] == 1]
        if event_rows.empty:
            continue
        event_score = event_rows.iloc[0]['score']
        first_score = g.iloc[0]['score']
        previous_scores = g.loc[g[TARGET_COL] == 0, 'score']
        max_prev = previous_scores.max() if len(previous_scores) else np.nan
        rows.append({
            ID_JORNADA_COL: jid,
            'n_simulacoes': len(g),
            'score_primeira': first_score,
            'score_proposta': event_score,
            'delta_score_proposta_primeira': event_score - first_score,
            'score_proposta_maior_que_primeira': event_score > first_score,
            'score_proposta_maior_que_anteriores': True if pd.isna(max_prev) else event_score > max_prev,
            'rank_proposta': int((g['score'] > event_score).sum() + 1),
        })
    return pd.DataFrame(rows)

# best_pw = metrics_all[(metrics_all.dataset_name == 'pointwise') & (metrics_all.split == 'oot')].sort_values('roc_auc', ascending=False).iloc[0]
# diag = journey_ranking_diagnostics(df_pw_split, pred_all, best_pw.model_name, split='oot')
# display(diag.describe(include='all'))
# display(diag.head())


## 19. Coeficientes e interpretação das features

In [ ]:
def get_pipeline_feature_names(model):
    try:
        if 'preprocess' in model.named_steps:
            return model.named_steps['preprocess'].get_feature_names_out()
        if 'woe' in model.named_steps:
            return model.named_steps['woe'].get_feature_names_out()
        if 'optbinning' in model.named_steps:
            return model.named_steps['optbinning'].get_feature_names_out()
        if 'treebin_woe' in model.named_steps:
            return model.named_steps['treebin_woe'].get_feature_names_out()
        if 'tree_encoder' in model.named_steps:
            n = model.named_steps['model'].coef_.shape[1]
            return np.array([f'tree_leaf_{i}' for i in range(n)])
    except Exception:
        pass
    n = model.named_steps['model'].coef_.shape[1]
    return np.array([f'feature_{i}' for i in range(n)])


def coefficient_table(model, top_n=40):
    coefs = model.named_steps['model'].coef_.ravel()
    names = get_pipeline_feature_names(model)
    n = min(len(coefs), len(names))
    out = pd.DataFrame({'feature': names[:n], 'coef': coefs[:n], 'abs_coef': np.abs(coefs[:n])})
    return out.sort_values('abs_coef', ascending=False).head(top_n)

# Exemplo:
# key = ('pointwise', best_pw.model_name)
# model = models_pw[key]
# display(coefficient_table(model, top_n=50))


## 20. Mapas de WOE para entender comportamento das features

In [ ]:
def show_woe_maps(model):
    if 'woe' in model.named_steps:
        transformer = model.named_steps['woe']
    elif 'treebin_woe' in model.named_steps:
        transformer = model.named_steps['treebin_woe']
    else:
        print('Modelo não possui transformer WOE simples/treebinning.')
        return None
    rows = []
    for feature, mp in transformer.maps_.items():
        for bin_value, woe in mp.items():
            rows.append({'feature': feature, 'bin_or_category': bin_value, 'woe': woe})
    return pd.DataFrame(rows).sort_values(['feature', 'woe'])

# Exemplo:
# woe_keys = [k for k in {**models_agg, **models_pw}.keys() if 'woe' in k[1]]
# display(show_woe_maps(({**models_agg, **models_pw})[woe_keys[0]]))


## 21. Export dos resultados

In [ ]:
# OUTPUT_DIR = '/mnt/data/pricing_pointwise_outputs'
# os.makedirs(OUTPUT_DIR, exist_ok=True)
# metrics_all.to_csv(f'{OUTPUT_DIR}/metrics_all.csv', index=False)
# pred_all.to_parquet(f'{OUTPUT_DIR}/predictions_all.parquet', index=False)
# print(f'Arquivos salvos em: {OUTPUT_DIR}')


## 22. Ordem recomendada de execução

1. Ajustar caminhos e nomes das colunas na configuração.
2. Carregar `df_raw`.
3. Rodar `check_required_columns(df_raw)`.
4. Construir `df_pointwise` e `df_agg`.
5. Aplicar split temporal.
6. Adicionar pesos.
7. Rodar primeiro `run_experiments(df_agg_split, 'agregado')`.
8. Rodar depois `run_experiments(df_pw_split, 'pointwise')`.
9. Comparar métricas no OOT.
10. Avaliar calibração e decil.
11. No pointwise, rodar o diagnóstico de jornada.
12. Abrir coeficientes e mapas de WOE dos melhores modelos.

Leitura recomendada:

- **OOT** deve ser o principal critério de comparação.
- **PR-AUC** e **KS** ajudam a avaliar ranking em evento raro.
- **Brier** e curvas de calibração mostram se o score pode ser usado como probabilidade.
- O diagnóstico de jornada responde se a simulação que virou proposta recebe score maior que as anteriores.
